<a href="https://colab.research.google.com/github/Alphaz-006/ml_project/blob/main/Speech_Emotion_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Speech Emotion Recognition

=== Setup ===



In [1]:
!pip install librosa kaggle -q
import os, librosa, numpy as np, pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [2]:
!kaggle datasets download -d uwrfkaggler/ravdess-emotional-speech-audio -p data --unzip


Dataset URL: https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio
License(s): CC-BY-NC-SA-4.0
100% 429M/429M [00:04<00:00, 91.2MB/s]



# === Feature extraction ===

In [3]:

def extract_mfcc(file_path, max_len=130):
    y, sr = librosa.load(file_path, sr=22050)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    if mfcc.shape[1] < max_len:
        mfcc = np.pad(mfcc, ((0,0),(0, max_len - mfcc.shape[1])))
    else:
        mfcc = mfcc[:, :max_len]
    return mfcc.T  # (time, features)



# === Build dataset ===

In [4]:

emotion_map = {'01':'neutral','02':'calm','03':'happy','04':'sad',
               '05':'angry','06':'fearful','07':'disgust','08':'surprised'}

X, y = [], []
root = "data"
for actor_dir in os.listdir(root):
    actor_path = os.path.join(root, actor_dir)
    if not os.path.isdir(actor_path): continue
    for f in os.listdir(actor_path):
        if f.endswith(".wav"):
            emotion_code = f.split("-")[2]
            X.append(extract_mfcc(os.path.join(actor_path, f)))
            y.append(emotion_map[emotion_code])


In [5]:
X = np.array(X)
le = LabelEncoder()
y_enc = le.fit_transform(y)
y_onehot = tf.keras.utils.to_categorical(y_enc)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y_onehot, test_size=0.2, random_state=42)

# === Model: CNN + LSTM ===

In [7]:

model = models.Sequential([
    layers.Input(shape=(X.shape[1], X.shape[2])),
    layers.Conv1D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),
    layers.Conv1D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),
    layers.LSTM(128, return_sequences=False),
    layers.Dropout(0.4),
    layers.Dense(64, activation='relu'),
    layers.Dense(y_onehot.shape[1], activation='softmax')
])



In [8]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 130, 64)        │         7,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 130, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 65, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 65, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 65, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 32, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 173,576 (678.03 KB)

 Trainable params: 173,192 (676.53 KB)

 Non-trainable params: 384 (1.50 KB)

In [9]:
history = model.fit(X_train, y_train, epochs=40, batch_size=32,
                     validation_data=(X_test, y_test))



Epoch 1/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 8s 97ms/step - accuracy: 0.3160 - loss: 1.8052 - val_accuracy: 0.2604 - val_loss: 1.9695
Epoch 2/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 74ms/step - accuracy: 0.4462 - loss: 1.4807 - val_accuracy: 0.3646 - val_loss: 1.8457
Epoch 3/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 70ms/step - accuracy: 0.5304 - loss: 1.3006 - val_accuracy: 0.3229 - val_loss: 1.8326
Epoch 4/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 133ms/step - accuracy: 0.5773 - loss: 1.1653 - val_accuracy: 0.3507 - val_loss: 1.8070
Epoch 5/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 89ms/step - accuracy: 0.6380 - loss: 0.9974 - val_accuracy: 0.4375 - val_loss: 1.6051
Epoch 6/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 87ms/step - accuracy: 0.6944 - loss: 0.8473 - val_accuracy: 0.5660 - val_loss: 1.2449
Epoch 7/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 90ms/step - accuracy: 0.7483 - loss: 0.7022 - val_accuracy: 0.5208 - val_loss: 1.5707
Epoch 8/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 4s 119ms/step - accuracy: 0.7873 - loss: 0.6287 - val_accuracy: 0.7014 -

# === Evaluate ===


In [10]:

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.2%}")



9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7743 - loss: 1.0369
Test Accuracy: 77.43%


===Saving the model===

In [11]:
model.save("speech_emotion_model.h5")